# 第五讲 — 预期与过渡动态 (Expectations and Transition Dynamics)

**异质性主体宏观经济学的计算方法**

孙杰


## 1. 准备工作

我们加载 `HouseholdStages`，以及绘图工具和 `Accessors`（用于比较静态分析中的 `@set` 宏）。


In [ ]:
using Pkg
Pkg.activate("..")
Pkg.instantiate()
using HouseholdStages, Plots, Printf, Accessors

## 2. `HouseholdStages` 中的 Aiyagari 模型

与第四讲相同的三阶段家庭问题，加上一个 Cobb-Douglas 厂商以及均衡条件 $\bar K = \bar K^{\mathrm{supplied}}$。全要素生产率（TFP）`A` 通过 `aiyagari_prices` 进入；家庭链的 env 只携带价格 `(r, w)`。

### 2.1 参数


In [ ]:
@kwdef struct AiyagariParams
    β::Float64 = 0.96
    σ::Float64 = 1.5
    α::Float64 = 0.36
    δ::Float64 = 0.08
    L::Float64 = 1.0
    A::Float64 = 1.0
    z_grid::Vector{Float64} = [0.6, 1.0, 1.4]
    P_z::Matrix{Float64}    = [0.7 0.2 0.1;
                               0.2 0.6 0.2;
                               0.1 0.2 0.7]
    N_w::Int       = 400
    w_min::Float64 = 0.0
    w_max::Float64 = 100.0
end

### 2.2 布局与家庭链

三个阶段，按时间顺序排列：

1. **马尔可夫生产率冲击 (Markov productivity shock)** — 沿 `:z` 轴的 `MarkovStage`。
2. **收入领取** — `WealthChangeStage`，对应期内预算恒等式 $w' = (1+r)\,w + w\cdot z$。
3. **消费-储蓄选择** — 采用 CRRA 效用的 `ConsumptionSavingsStage`。

用 `∘` 组合它们（时间顺序：最左侧的先运行），然后 `define_moments!` 附加外层 tatonnement 所需的总财富积分矩。


In [ ]:
aiyagari_layout(p) = StateLayout(
    StateAxis(:wealth, continuous_grid(p.w_min, p.w_max; length=p.N_w, spacing=:log)),
    StateAxis(:z, p.z_grid),
)

_u_crra(c, ::Val{1}) = log(c)
_u_crra(c, ::Val{σ}) where σ = (c^(1 - σ)) / (1 - σ)
u_crra(c, valσ) = c < 0 ? -Inf : _u_crra(c, valσ)

function aiyagari_household(p::AiyagariParams)
    layout = aiyagari_layout(p)

    # 阶段 1 —— 马尔可夫生产率冲击
    z_shock = MarkovStage(layout; axis=:z, transition=p.P_z)

    # 阶段 2 —— 收入领取
    income = WealthChangeStage(layout; wealth_post=(cell; env) -> (1 + env.r) * cell.wealth + env.w * cell.z)

    # 阶段 3 —— 消费-储蓄
    savings = ConsumptionSavingsStage(layout; β=p.β, utility=(cell, c; env) -> u_crra(c, Val(p.σ)), monotone_search=:divide_conquer)

    # 按时间顺序组合 —— 最左侧的先运行
    hh = z_shock ∘ income ∘ savings

    # 附加 K_supplied 矩（链末端的总财富积分）
    return define_moments!(hh; K_supplied=at_end(integrand=:wealth, reduce=sum))
end

# Cobb-Douglas 要素价格；A 从 p 中读出，所以 env 仍是 (r, w)。
function aiyagari_prices(K, p::AiyagariParams)
    (; α, δ, L, A) = p
    return (; r=A*α*(K/L)^(α-1) - δ, w=A*(1-α)*(K/L)^α)
end

### 2.3 `hh` 长什么样？

一个打包好的 `ChainStage` 携带一个 Spec（纯配置：布局、转移矩阵、闭包、附加的矩）和一个 Buffer（每次调用的状态：核、临时数组、热启动 V/Λ、核缓存）。用户从不直接接触 Buffer —— 他们只需把 `hh` 传给软件包的辅助函数。


In [ ]:
hh = aiyagari_household(AiyagariParams())
dump(hh; maxdepth=1)

### 2.4 单 env 探针

"在单个 env 下求解"对外暴露的接口是 `solve_steady_state_given_env!(hh, env)`。它运行 V 的后向迭代和 Λ 的前向迭代，返回 `V`、`Λ` 以及已定义 `moments` 的副本 —— 并顺便为链的 buffer 热启动下一次调用。

用 `make_env(hh; ...)` 来构造 env。这个辅助函数会根据链的 schema 验证字段名，若缺失任何必需键，会抛出清晰的错误。


In [ ]:
p   = AiyagariParams()
hh  = aiyagari_household(p)
env = make_env(hh; aiyagari_prices(5.0, p)...)
res = solve_steady_state_given_env!(hh, env)

@printf "K_supplied = %.4f  (K_guess = 5.0)\n" res.moments.K_supplied
@printf "VFI iters: %d   Λ iters: %d\n" res.history.vfi_iters res.history.lambda_iters

### 2.5 通过对 K 进行 tatonnement 求稳态

外层循环：猜测 $K$，从家庭模块得到 $K^{\mathrm{supplied}}$，对 $K$ 做阻尼更新，重复。链的 buffer 自动热启动每次调用，因此随着外层迭代推进，内层求解会越来越便宜。


In [ ]:
function aiyagari_steady_state(p::AiyagariParams; K_init=5.0, update_speed=0.01)
    hh = aiyagari_household(p)
    K  = K_init
    local V, Λ
    history = Float64[]
    for it in 1:500
        env = make_env(hh; aiyagari_prices(K, p)...)
        (;V, Λ, moments) = solve_steady_state_given_env!(hh, env)
        K_S = moments.K_supplied
        err = abs(K_S - K) / K
        push!(history, err)
        err ≤ 2e-2 && return (; K, V, Λ, hh, history, iters=it)
        K += update_speed * (K_S - K)
    end
    error("aiyagari_steady_state: did not converge")
end

ss = aiyagari_steady_state(p)
@printf "K_ss = %.4f, r = %.4f, w = %.4f  (in %d outer iters)\n" ss.K aiyagari_prices(ss.K, p).r aiyagari_prices(ss.K, p).w ss.iters

### 2.6 比较静态分析

永久性的 5% 正向 TFP 冲击。我们用 `@set` 替换 `p.A`，并从旧稳态出发重新运行 tatonnement（这是一个好的热启动）。


In [ ]:
ss_old = aiyagari_steady_state(p)
p_new  = @set p.A = 1.05
ss_new = aiyagari_steady_state(p_new; K_init=ss_old.K * 1.05)

@printf "ΔK = %+0.4f  (%.2f%%)\n" (ss_new.K - ss_old.K) 100*(ss_new.K/ss_old.K - 1)
@printf "old K_ss = %.4f   new K_ss = %.4f\n" ss_old.K ss_new.K

## 3. MIT 冲击 (MIT shock) — 完美预见过渡 (perfect-foresight transition)

**给定。** 外生路径 $\{A_t\}_{t=1}^T$、家庭链 `hh`、厂商参数 $(\alpha, \delta, L)$。

**求。** 序列 $\{K_t\}, \{V_t\}, \{\Lambda_t\}$，满足：

- $V_{T+1} = V_{\mathrm{ss\_new}}$（终端条件），
- $\Lambda_1 = \Lambda_{\mathrm{ss\_old}}$（初始条件），
- $V_t = \texttt{backward!}(V_{t+1}, \mathrm{env}_t)$，$t = T, T{-}1, \ldots, 1$，
- $\Lambda_{t+1} = \texttt{forward!}(\Lambda_t, V_{t+1}, \mathrm{env}_t)$，$t = 1, 2, \ldots, T$，
- $K_t = \int b\,\mathrm{d}\Lambda_{t+1} = K_t^{\mathrm{supplied}}$（市场每期出清）。

**算法。** 猜测 $\{K_t\}$，从终端条件出发后向扫一遍 $V$，从初始条件出发前向扫一遍 $\Lambda$，读出 $K_t^{\mathrm{supplied}}$，对 $\{K_t\}$ 做阻尼更新。

### 3.1 简洁版本 —— 使用 `solve_transition_given_env_path!`

`solve_transition_given_env_path!` 负责每期 buffer 的分配、后向与前向扫描（含核缓存的正确再绑定），以及每期的矩计算。外层循环只需提供 env 路径和边界条件。


In [ ]:
function mit_shock_transition(p::AiyagariParams; A_new=1.05, T=100, update_speed=0.2, tol=1e-3, max_iter=200, verbose=false)
    # 1-2. 端点稳态
    pre   = aiyagari_steady_state(p)
    p_new = @set p.A = A_new
    post  = aiyagari_steady_state(p_new; K_init=pre.K * A_new)

    hh      = aiyagari_household(p_new)
    K_path  = collect(range(pre.K, post.K; length=T))
    history = Float64[]

    for it in 1:max_iter
        # 3a. 根据当前 K 猜测构造 env 路径
        env_path = [make_env(hh; aiyagari_prices(K_path[t], p_new)...) for t in 1:T]

        # 3b. 一次后向 + 一次前向扫描，边界条件显式传入
        (;V_path, Λ_path, moments_path) = solve_transition_given_env_path!(hh, env_path; Λ_0=pre.Λ, V_T=post.V)

        # 3c. 残差 + 阻尼更新
        K_S = getproperty.(moments_path, :K_supplied)
        err = maximum(abs.(K_S .- K_path))
        push!(history, err)
        verbose && (it ≤ 5 || it % 5 == 0) &&
            @printf "  iter %3d: ‖K^S − K‖∞ = %.4e\n" it err

        err ≤ tol && return (; K_path, K_S, V_path, Λ_path, pre, post, history, iters=it)
        K_path .= (1 - update_speed) .* K_path .+ update_speed .* K_S
    end
    error("mit_shock_transition: did not converge")
end

T  = 100
tr = mit_shock_transition(p; A_new=1.05, T=T, verbose=true)
@printf "\nConverged in %d outer iters.\n" tr.iters
@printf "K_ss_pre  = %.4f\n" tr.pre.K
@printf "K_ss_post = %.4f\n" tr.post.K
@printf "K[1]   (impact) = %.4f\n" tr.K_path[1]
@printf "K[5]            = %.4f\n" tr.K_path[5]
@printf "K[20]           = %.4f\n" tr.K_path[20]
@printf "K[end] (≈post)  = %.4f\n" tr.K_path[end]

### 3.2 手动版本 —— `solve_transition_given_env_path!` 内部是什么

同样的算法，每一步都展开。这在教学上有用：简洁版本*就是*这个循环，只是被打包了。

该版本暴露了三件事：

- **每期一条链** —— `hh_path[t]` 是每期一条链，共享 Spec 但各自拥有 Buffer。这样每期的后向结果（核）就被保留下来供匹配的前向扫描使用，无需重做。一条链也能工作 —— 核缓存会在 `forward!` 看到不匹配的 `V_end` 时通过重新运行 `backward!` 来再绑定 —— 但这会使每次迭代的工作量翻倍。
- **边界条件作为数组端点** —— `V_path[T+1] = post.V` 和 `Λ_path[1] = pre.Λ` 位于数组的端点，直接取自稳态求解的返回值。没有访问链内部的访问器。
- **`backward!` 和 `forward!` 返回它们的输出** —— `V_path[t]` 由 `backward!` 流出，`Λ_path[t+1]` 由 `forward!` 流出。每期的矩通过 `compute_moments(hh, Λ, env)` 计算，`Λ` 显式传入。


In [ ]:
function mit_shock_transition_manual(p::AiyagariParams; A_new=1.05, T=100, update_speed=0.2)
    # 端点稳态
    pre   = aiyagari_steady_state(p)
    p_new = @set p.A = A_new
    post  = aiyagari_steady_state(p_new; K_init=pre.K * A_new)

    # 每期一条链 —— 同一个 spec，全新的 buffer。
    hh_path = [aiyagari_household(p_new) for _ in 1:T]
    dims    = layout_size(aiyagari_layout(p_new))

    # V_path[t]   = 第 t 期开始时的延续价值
    # V_path[T+1] = post.V    （终端边界）
    # Λ_path[t]   = 第 t 期开始时的分布
    # Λ_path[1]   = pre.Λ     （初始边界）
    V_path = [zeros(Float64, dims...) for _ in 1:T+1]
    Λ_path = [zeros(Float64, dims...) for _ in 1:T+1]
    copyto!(V_path[T+1], post.V)
    copyto!(Λ_path[1],   pre.Λ)

    K_path  = collect(range(pre.K, post.K; length=T))
    history = Float64[]

    for it in 1:200
        env_path = [make_env(hh_path[t]; aiyagari_prices(K_path[t], p_new)...) for t in 1:T]

        # 后向扫描：在链 hh_path[t] 上执行 V_t = backward!(V_{t+1}, env_t)。
        # backward! 返回 V_start；将其复制到 V_path[t]。
        for t in T:-1:1
            copyto!(V_path[t], backward!(hh_path[t], V_path[t+1], env_path[t]))
        end

        # 前向扫描：在链 hh_path[t] 上执行 Λ_{t+1} = forward!(Λ_t)。
        # 核缓存是新鲜的（我们刚在该链上跑过 backward），所以
        # 廉价的 forward 调用按构造即正确。
        K_S = zeros(T)
        for t in 1:T
            copyto!(Λ_path[t+1], forward!(hh_path[t], Λ_path[t]))
            K_S[t] = compute_moments(hh_path[t], Λ_path[t+1], env_path[t]).K_supplied
        end

        err = maximum(abs.(K_S .- K_path))
        push!(history, err)

        err ≤ 1e-3 && return (; K_path, K_S, V_path, Λ_path, pre, post, history, iters=it)
        K_path .= (1 - update_speed) .* K_path .+ update_speed .* K_S
    end
    error("mit_shock_transition_manual: did not converge")
end

tr_manual = mit_shock_transition_manual(p; A_new=1.05, T=T)
@printf "Manual converged in %d outer iters.\n" tr_manual.iters
@printf "‖K_path_clean − K_path_manual‖∞ = %.2e\n" maximum(abs.(tr.K_path .- tr_manual.K_path))

### 3.3 资本脉冲响应 (Capital IRF)

+5% TFP 冲击后的总资本路径，前后稳态以虚线和点线水平线标注。


In [ ]:
plot(1:T, tr.K_path; lw=2, label="K_t",
     xlabel="period t", ylabel="aggregate capital K",
     title="IRF: K to a +5% permanent TFP shock")
hline!([tr.pre.K];  color=:gray, linestyle=:dash, label="K_ss^pre")
hline!([tr.post.K]; color=:gray, linestyle=:dot,  label="K_ss^post")

### 3.4 实际利率与工资路径

使用新的（冲击后）TFP 参数，通过 `aiyagari_prices` 从 K 路径读出价格。


In [ ]:
prices_path = [aiyagari_prices(K, p_new) for K in tr.K_path]
r_path = getproperty.(prices_path, :r)
w_path = getproperty.(prices_path, :w)

plot(layout=(2, 1), size=(700, 500))
plot!(1:T, r_path; subplot=1, lw=2, label="r_t",
      xlabel="period", ylabel="r", title="IRF: real rate")
hline!([aiyagari_prices(tr.pre.K,  p     ).r]; subplot=1, color=:gray, linestyle=:dash, label="r_ss^pre")
hline!([aiyagari_prices(tr.post.K, p_new ).r]; subplot=1, color=:gray, linestyle=:dot,  label="r_ss^post")

plot!(1:T, w_path; subplot=2, lw=2, label="w_t",
      xlabel="period", ylabel="w", title="IRF: wage")
hline!([aiyagari_prices(tr.pre.K,  p     ).w]; subplot=2, color=:gray, linestyle=:dash, label="w_ss^pre")
hline!([aiyagari_prices(tr.post.K, p_new ).w]; subplot=2, color=:gray, linestyle=:dot,  label="w_ss^post")

### 3.5 Tatonnement 残差历史

阻尼 tatonnement 让残差几何下降，直到在基准校准下触及 $\sim 2.5\times 10^{-3}$ 的*离散化下限*。这个下限来源于硬 `argmax` 的 `ConsumptionSavingsStage` 政策，在 $K_t$ 摆动时在相邻网格点之间翻转。平滑的（基于 `LogitChoiceStage` 的）储蓄政策或更密的财富网格能把下限压低。


In [ ]:
plot(1:length(tr.history), tr.history;
     yscale=:log10, lw=2, marker=:circle, markersize=3,
     xlabel="outer iteration", ylabel="‖K^S − K‖∞",
     label="residual", title="Damped tatonnement residual history")

### 3.6 更新速度扫描

更新速度是一门手艺：太高会振荡，太低会爬行。在 $s \in \{0.1, 0.2, 0.4, 0.6\}$ 下重新运行过渡，并叠加残差历史。

**预期。** $s = 0.6$ 会振荡或无法收敛；$s = 0.1$ 收敛缓慢；$s = 0.2$–$0.4$ 大致是甜蜜点。


In [ ]:
plt = plot(yscale=:log10, xlabel="outer iteration",
           ylabel="residual ‖K^S − K‖∞",
           title="Update-speed sweep")
for s in (0.1, 0.2, 0.4, 0.6)
    local r
    try
        r = mit_shock_transition(p; A_new=1.05, T=T, update_speed=s, tol=1e-3, max_iter=80)
    catch err
        # 若某个设置在 max_iter 内未收敛，函数会报错；
        # 我们仍想绘制它已有的残差历史。
        @warn "s = $s did not converge; plotting partial history."
        continue
    end
    plot!(plt, 1:length(r.history), r.history; lw=2, label="s = $s")
end
plt